# CSP — Constraint Satisfaction Problem
### Meal Planner using AC-3 + Backtracking
> Uses `TransitionModel` directly — consistent with GA, A*, and Greedy modules.

In [34]:
import random
import time
import pandas as pd

# ─── YOUR MEAL DATABASE (from CSV) ───────────────────────────────────────
meals_df = pd.read_csv('data/recipes.csv')
meals_db = []
for _, row in meals_df.iterrows():
    meals_db.append({
        "name": row["Name"],
        "type": row["type"],
        "kcal": row["provided_calories"],
        "cost": row["total_price"],
        "category": row["Category"],
        "protein_g": row["provided_protein"],
        "carbs_g": row["provided_carbs"],
        "fat_g": row["provided_fat"]
    })

class TransitionModel:
    def __init__(self, meals_db):
        self.model = {}
        for meal in meals_db:
            meal_type = meal["type"]
            if meal_type not in self.model:
                self.model[meal_type] = {}
            # Store as (category, cost, kcal, protein, carbs, fat)
            self.model[meal_type][meal["name"]] = (
                meal["category"],
                meal["cost"],
                meal["kcal"],
                meal["protein_g"],
                meal["carbs_g"],
                meal["fat_g"]
            )

# Create global instance for compatibility with existing code
TransitionModel = TransitionModel(meals_db).model

# User Input

In [35]:
TDEE         = 2000    # target daily calories
DAILY_BUDGET = 1500    # DZD per day
PREFERENCE   = "Non-Vegetarian"   # "Vegetarian" or "Non-Vegetarian"
DAYS         = 30
GOAL         = "maintenance"      # "weight_loss", "muscle_gain", "maintenance"

NUTRI_PREFERENCE_MAP = {
    "maintenance":  {"protein": 0.30, "carbs": 0.40, "fat": 0.30},
    "weight_loss":  {"protein": 0.40, "carbs": 0.30, "fat": 0.30},
    "muscle_gain":  {"protein": 0.40, "carbs": 0.40, "fat": 0.20},
}

# Macro Delta
Reads directly from `TransitionModel` — no separate meal dicts needed.

In [36]:
def compute_macro_delta(day_meal_names, goal="maintenance"):
    """
    day_meal_names: tuple/list of (breakfast_name, lunch_name, dinner_name)
    Reads nutrition from TransitionModel directly.
    Returns dict with actual, target, delta per macro, and a 0-1 penalty score.
    """
    slots  = ["Breakfast", "Lunch", "Dinner"]
    ratios = NUTRI_PREFERENCE_MAP[goal]

    actual_protein = sum(TransitionModel[slots[i]][day_meal_names[i]][3] for i in range(3))
    actual_carbs   = sum(TransitionModel[slots[i]][day_meal_names[i]][4] for i in range(3))
    actual_fat     = sum(TransitionModel[slots[i]][day_meal_names[i]][5] for i in range(3))

    # Convert calorie-ratio targets to grams
    target_protein_g = (TDEE * ratios["protein"]) / 4
    target_carbs_g   = (TDEE * ratios["carbs"])   / 4
    target_fat_g     = (TDEE * ratios["fat"])      / 9

    delta_protein = actual_protein - target_protein_g
    delta_carbs   = actual_carbs   - target_carbs_g
    delta_fat     = actual_fat     - target_fat_g

    penalty = (
        abs(delta_protein) / target_protein_g +
        abs(delta_carbs)   / target_carbs_g   +
        abs(delta_fat)     / target_fat_g
    ) / 3

    return {
        "actual": {"protein_g": actual_protein, "carbs_g": actual_carbs, "fat_g": actual_fat},
        "target": {"protein_g": target_protein_g, "carbs_g": target_carbs_g, "fat_g": target_fat_g},
        "delta":  {"protein_g": delta_protein,  "carbs_g": delta_carbs,  "fat_g": delta_fat},
        "penalty": min(penalty, 1.0)
    }

# Domain Building

In [37]:
def build_domains():
    """
    Builds initial domains from TransitionModel.
    Each domain entry is a meal_name string (consistent with TransitionModel keys).
    Pre-filters by: preference, per-slot calorie cap, and per-slot budget cap.
    """
    slot_calorie_share = {"Breakfast": 0.35, "Lunch": 0.40, "Dinner": 0.40}
    slot_budget_share  = {"Breakfast": 0.35, "Lunch": 0.40, "Dinner": 0.40}

    domains = {}
    for day in range(DAYS):
        for slot in ["Breakfast", "Lunch", "Dinner"]:
            cal_cap    = TDEE         * slot_calorie_share[slot]
            budget_cap = DAILY_BUDGET * slot_budget_share[slot]

            domains[(day, slot)] = [
                meal_name
                for meal_name, info in TransitionModel[slot].items()
                # info = (Category, cost, kcal, protein, carbs, fat)
                if (PREFERENCE != "Vegetarian" or info[0] == "Vegetarian")
                and info[2] <= cal_cap       # kcal within slot share
                and info[1] <= budget_cap    # cost within slot share
            ]
    return domains

# Constraints
Arc compatibility check: calories + budget.

In [38]:
def are_compatible(slot_i, meal_i_name, slot_j, meal_j_name,
                   third_min_kcal, third_max_kcal, third_min_cost):
    """
    Returns True if meal_i and meal_j can coexist on the same day,
    meaning there exists at least one third meal that satisfies:
      - daily kcal within TDEE +-10%
      - daily cost <= DAILY_BUDGET
    """
    info_i = TransitionModel[slot_i][meal_i_name]
    info_j = TransitionModel[slot_j][meal_j_name]

    combined_kcal = info_i[2] + info_j[2]
    combined_cost = info_i[1] + info_j[1]

    low  = TDEE * 0.9
    high = TDEE * 1.1

    needed_kcal_min = low  - combined_kcal
    needed_kcal_max = high - combined_kcal

    calorie_ok = (needed_kcal_max >= third_min_kcal) and (needed_kcal_min <= third_max_kcal)
    budget_ok  = (combined_cost + third_min_cost) <= DAILY_BUDGET

    return calorie_ok and budget_ok

# AC-3 Algorithm

In [39]:
def ac3(domains):
    """
    Enforces arc consistency across all (day, slot) pairs.
    Prunes meal names from domains that can never be part of a valid day plan.
    Returns True if domains are non-empty (solution may exist), False otherwise.
    """
    slots = ["Breakfast", "Lunch", "Dinner"]
    queue = [
        (day, s1, s2)
        for day in range(DAYS)
        for s1 in slots
        for s2 in slots
        if s1 != s2
    ]

    while queue:
        day, slot_i, slot_j = queue.pop(0)

        third_slot = [s for s in slots if s != slot_i and s != slot_j][0]
        third_infos = [TransitionModel[third_slot][m] for m in domains[(day, third_slot)]]

        if not third_infos:
            return False

        third_min_kcal = min(info[2] for info in third_infos)
        third_max_kcal = max(info[2] for info in third_infos)
        third_min_cost = min(info[1] for info in third_infos)

        revised = False
        for meal_i in domains[(day, slot_i)][:]:   # iterate over copy
            compatible_exists = any(
                are_compatible(
                    slot_i, meal_i, slot_j, meal_j,
                    third_min_kcal, third_max_kcal, third_min_cost
                )
                for meal_j in domains[(day, slot_j)]
            )
            if not compatible_exists:
                domains[(day, slot_i)].remove(meal_i)
                revised = True

        if revised:
            if len(domains[(day, slot_i)]) == 0:
                return False  # domain wiped -> no solution
            for slot_k in slots:
                if slot_k != slot_i:
                    queue.append((day, slot_k, slot_i))

    return True

# Backtracking Search

In [40]:
def backtrack_day(domains, day):
    """
    Backtracking search for one day.
    Returns dict {"Breakfast": meal_name, "Lunch": meal_name, "Dinner": meal_name}
    or None if no valid combo exists.
    """
    slots = ["Breakfast", "Lunch", "Dinner"]
    low   = TDEE * 0.9
    high  = TDEE * 1.1

    def backtrack(index, used_kcal, used_cost, assignment):
        if index == len(slots):
            if low <= used_kcal <= high and used_cost <= DAILY_BUDGET:
                return dict(assignment)
            return None

        slot = slots[index]
        for meal_name in random.sample(domains[(day, slot)], len(domains[(day, slot)])):
            info     = TransitionModel[slot][meal_name]
            new_kcal = used_kcal + info[2]
            new_cost = used_cost + info[1]

            if new_cost > DAILY_BUDGET or new_kcal > high:
                continue

            assignment[slot] = meal_name
            result = backtrack(index + 1, new_kcal, new_cost, assignment)
            if result:
                return result
            del assignment[slot]

        return None

    return backtrack(0, 0, 0, {})


def solve(domains):
    """Run backtracking for all 30 days. Returns full plan or None."""
    plan = {}
    for day in range(DAYS):
        day_plan = backtrack_day(domains, day)
        if day_plan is None:
            return None
        plan[day] = day_plan
    return plan

# Execution

In [41]:
print("=" * 50)
print("        CSP Meal Planner - 30 Days")
print("=" * 50)
print(f"TDEE: {TDEE} kcal | Budget: {DAILY_BUDGET} DZD/day | Goal: {GOAL} | Preference: {PREFERENCE}\n")

t0 = time.time()
domains = build_domains()
t1 = time.time()
print(f"[1] Domain building:  {t1-t0:.4f}s")
print(f"    Sizes before AC-3 (day 0): { {s: len(domains[(0,s)]) for s in ['Breakfast','Lunch','Dinner']} }")

success = ac3(domains)
t2 = time.time()
print(f"\n[2] AC-3:             {t2-t1:.4f}s  |  success={success}")
print(f"    Sizes after AC-3  (day 0): { {s: len(domains[(0,s)]) for s in ['Breakfast','Lunch','Dinner']} }")

if success:
    plan = solve(domains)
    t3 = time.time()
    print(f"\n[3] Backtracking:     {t3-t2:.4f}s")
    print(f"    Total time:       {t3-t0:.4f}s")

    if plan:
        print("\nSolution found!\n")

        total_cost = total_kcal = 0
        total_macro_penalty = 0

        for day in range(DAYS):
            bp = plan[day]["Breakfast"]
            lp = plan[day]["Lunch"]
            dp = plan[day]["Dinner"]

            day_cost = (TransitionModel["Breakfast"][bp][1] +
                        TransitionModel["Lunch"][lp][1]    +
                        TransitionModel["Dinner"][dp][1])
            day_kcal = (TransitionModel["Breakfast"][bp][2] +
                        TransitionModel["Lunch"][lp][2]    +
                        TransitionModel["Dinner"][dp][2])

            total_cost += day_cost
            total_kcal += day_kcal

            macro = compute_macro_delta((bp, lp, dp), GOAL)
            total_macro_penalty += macro["penalty"]

        avg_macro_penalty = total_macro_penalty / DAYS

        print(f"Total cost        : {total_cost:.2f} DZD  (budget: {DAILY_BUDGET*DAYS} DZD)")
        print(f"Average daily cost: {total_cost/DAYS:.2f} DZD")
        print(f"Total calories    : {total_kcal:.0f} kcal")
        print(f"Avg daily calories: {total_kcal/DAYS:.1f} kcal  (target: {TDEE})")
        print(f"Avg macro penalty : {avg_macro_penalty:.3f}  (0=perfect, 1=worst)")

        # First 3 days detail
        print("\n--- First 3 Days ---")
        for day in range(min(3, DAYS)):
            bp = plan[day]["Breakfast"]
            lp = plan[day]["Lunch"]
            dp = plan[day]["Dinner"]
            macro = compute_macro_delta((bp, lp, dp), GOAL)

            day_kcal = (TransitionModel["Breakfast"][bp][2] +
                        TransitionModel["Lunch"][lp][2]    +
                        TransitionModel["Dinner"][dp][2])
            day_cost = (TransitionModel["Breakfast"][bp][1] +
                        TransitionModel["Lunch"][lp][1]    +
                        TransitionModel["Dinner"][dp][1])

            print(f"\nDay {day+1}:  {day_kcal:.0f} kcal | {day_cost:.2f} DZD")
            print(f"  Breakfast : {bp}")
            print(f"  Lunch     : {lp}")
            print(f"  Dinner    : {dp}")
            print(f"  Macros -> Protein: {macro['actual']['protein_g']:.1f}g "
                  f"(target {macro['target']['protein_g']:.1f}g) | "
                  f"Carbs: {macro['actual']['carbs_g']:.1f}g "
                  f"(target {macro['target']['carbs_g']:.1f}g) | "
                  f"Fat: {macro['actual']['fat_g']:.1f}g "
                  f"(target {macro['target']['fat_g']:.1f}g)")
            print(f"  Macro penalty: {macro['penalty']:.3f}")

        CSP Meal Planner - 30 Days
TDEE: 2000 kcal | Budget: 1500 DZD/day | Goal: maintenance | Preference: Non-Vegetarian

[1] Domain building:  0.0010s
    Sizes before AC-3 (day 0): {'Breakfast': 57, 'Lunch': 42, 'Dinner': 41}

[2] AC-3:             0.1070s  |  success=True
    Sizes after AC-3  (day 0): {'Breakfast': 50, 'Lunch': 35, 'Dinner': 29}

[3] Backtracking:     0.0060s
    Total time:       0.1140s

Solution found!

Total cost        : 19544.70 DZD  (budget: 45000 DZD)
Average daily cost: 651.49 DZD
Total calories    : 55027 kcal
Avg daily calories: 1834.2 kcal  (target: 2000)
Avg macro penalty : 1.000  (0=perfect, 1=worst)

--- First 3 Days ---

Day 1:  1887 kcal | 523.98 DZD
  Breakfast : Banana Almond Raib Bowl
  Lunch     : Chicken Chickpea Bowl
  Dinner    : Dolma
  Macros -> Protein: 498.5g (target 150.0g) | Carbs: 828.5g (target 200.0g) | Fat: 588.5g (target 66.7g)
  Macro penalty: 1.000

Day 2:  1820 kcal | 597.92 DZD
  Breakfast : Zucchini Herb Omelette
  Lunch   

# Analysis

In [42]:
if plan:
    print("\n--- Variety Analysis ---")
    meal_counts = {}
    for day_plan in plan.values():
        for meal_name in day_plan.values():
            meal_counts[meal_name] = meal_counts.get(meal_name, 0) + 1

    unique = len(meal_counts)
    top5   = sorted(meal_counts.items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"Unique meals used: {unique}")
    print("Top 5 most repeated:")
    for name, count in top5:
        print(f"  {name}: {count}x")

    violations = sum(
        1 for day in range(DAYS)
        if not (TDEE * 0.9 <=
                sum(TransitionModel[s][plan[day][s]][2] for s in ["Breakfast","Lunch","Dinner"])
                <= TDEE * 1.1)
        or sum(TransitionModel[s][plan[day][s]][1] for s in ["Breakfast","Lunch","Dinner"]) > DAILY_BUDGET
    )
    print(f"\nDays with constraint violations: {violations}")
else:
    print("Backtracking found no solution after AC-3.")


--- Variety Analysis ---
Unique meals used: 46
Top 5 most repeated:
  Dolma: 8x
  Lham Lahlou: 7x
  Harira: 6x
  Chicken Chickpea Bowl: 5x
  Green Bean Rice Bowl: 5x

Days with constraint violations: 0
